# Notebook 04 — Decision Trees and Random Forests

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 04 of 15  
**Time estimate:** 90 minutes

> Random forests are the most reliable ML model for tabular biological data.
> They are non-parametric, robust to outliers, handle mixed feature types,
> give built-in feature importance, and work without feature scaling.
> If you only learn one ML model for biology, learn this one.

---
## Step 1 — Motivation

Clinical data is tabular: age, tumor stage, mutation status, gene expression levels.
Features are mixed (continuous, categorical, ordinal), distributions are non-Gaussian,
and sample sizes are moderate (hundreds to thousands). Random forests handle all of
this gracefully and require minimal preprocessing.

---
## Step 2 — Intuition

**Decision tree:** Recursively partition the feature space by asking binary questions
("Is gene A expression > 3.2?"). Each split maximizes class purity.

**Problem with a single tree:** High variance — small changes in training data produce
very different trees. A single deep tree overfits badly.

**Random forest:** Train many trees, each on a bootstrap sample of the training data,
each using a random subset of features at each split. Average (regression) or vote
(classification) across trees.

**Why it works — two sources of randomness:**
1. **Bootstrap sampling (bagging):** Each tree sees a different random subset of the
   training data → trees are decorrelated
2. **Feature subsampling:** At each node, only $\sqrt{p}$ (classification) or $p/3$
   (regression) features are considered → prevents one dominant feature from appearing
   in every tree

Result: variance is reduced (averaging), bias stays similar to a single tree,
ensemble error $\approx \rho \sigma^2 + (1-\rho)\sigma^2/T$ where $\rho$ is inter-tree
correlation and $T$ is the number of trees.

---
## Step 3 — Biological Background

**Feature importance in biology:** Random forest feature importance (mean decrease
in impurity, MDI) identifies which genes/mutations are most informative.
Caveat: MDI is biased toward high-cardinality continuous features — permutation
importance (Breiman 2001) is more reliable.

**Out-of-bag (OOB) error:** Because each tree uses only ~63% of training data, the
remaining ~37% (out-of-bag samples) provide a free internal validation estimate —
useful when the dataset is too small for a separate validation set.

**Applications in biology:**
- **GWAS/variant effect:** Random forests predict variant pathogenicity from
  conservation, MAF, functional annotation (CADD uses an ensemble approach)
- **Clinical outcome prediction:** Combining clinical + genomic features for
  prognosis (survival, treatment response)
- **Cell type classification:** scRNA-seq clusters assigned labels from a reference;
  RF trained on labeled reference, applied to query data

**Interpretability:** Decision trees are directly interpretable (rule extraction).
Random forests are not (ensemble of ~100 trees). Use SHAP values for interpretation.

---
## Step 4 — Mathematical Explanation

**Information gain (entropy-based splitting criterion):**
$$H(S) = -\sum_{k} p_k \log_2 p_k$$

For binary split: $\text{IG}(S, A) = H(S) - \frac{|S_L|}{|S|}H(S_L) - \frac{|S_R|}{|S|}H(S_R)$

**Gini impurity:**
$$G(S) = 1 - \sum_k p_k^2$$

For balanced binary: $G = 0.5$ (maximum impurity). Gini and entropy give nearly
identical splits in practice; Gini is faster to compute (no log).

**Stopping criteria:** Max depth, min samples per leaf, min impurity decrease.

**Bagging variance reduction:** If $T$ independent trees each have variance $\sigma^2$,
their average has variance $\sigma^2/T$. With correlated trees (correlation $\rho$):
$$\text{Var}(\bar{f}) = \rho \sigma^2 + \frac{1-\rho}{T} \sigma^2$$

Feature subsampling reduces $\rho$ → lower ensemble variance.

In [ ]:
# Step 6 — Python: Decision tree from scratch + Random Forest via sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# ---- Decision tree node: information gain from scratch ----
def entropy(y):
    if len(y) == 0: return 0.0
    classes, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return -np.sum(p * np.log2(p + 1e-12))

def gini(y):
    if len(y) == 0: return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p**2)

def information_gain(y_parent, y_left, y_right, criterion='gini'):
    fn = gini if criterion == 'gini' else entropy
    n = len(y_parent)
    nL, nR = len(y_left), len(y_right)
    return fn(y_parent) - (nL/n)*fn(y_left) - (nR/n)*fn(y_right)

def best_split(X, y, criterion='gini'):
    """
    Find the best (feature, threshold) split by exhaustive search.
    Returns: (best_feature, best_threshold, best_ig)
    """
    n, p = X.shape
    best_ig, best_feat, best_thresh = -1, 0, 0
    for feat in range(p):
        thresholds = np.unique(X[:, feat])
        for t in thresholds[:-1]:
            left = y[X[:, feat] <= t]
            right = y[X[:, feat] > t]
            ig = information_gain(y, left, right, criterion)
            if ig > best_ig:
                best_ig, best_feat, best_thresh = ig, feat, t
    return best_feat, best_thresh, best_ig

# ---- Generate clinical tabular dataset ----
rng = np.random.default_rng(42)
n_patients = 300
# Features: age, tumor_size, mutation_count, gene_expr_A, gene_expr_B
age = rng.normal(60, 12, n_patients).clip(20, 90)
tumor_size = rng.exponential(2.5, n_patients).clip(0.1, 10)
mut_count = rng.poisson(3, n_patients)
gene_A = rng.normal(5, 2, n_patients)
gene_B = rng.normal(5, 2, n_patients)

# True label: bad prognosis if tumor_size>3 AND gene_A>6, OR mut_count>5
bad_prognosis = ((tumor_size > 3) & (gene_A > 6)) | (mut_count > 5)
y = bad_prognosis.astype(int)
X = np.column_stack([age, tumor_size, mut_count, gene_A, gene_B])
feature_names = ['Age', 'Tumor size', 'Mut count', 'Gene A', 'Gene B']
print(f'Dataset: {n_patients} patients, {y.mean():.1%} bad prognosis')

# ---- Verify best_split on a 2-feature subset ----
best_f, best_t, best_ig = best_split(X[:, 1:3], y)  # tumor_size and mut_count
print(f'Best split: feature {["Tumor size","Mut count"][best_f]}, threshold={best_t:.3f}, IG={best_ig:.4f}')

# ---- Random Forest via sklearn ----
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

results = {}
for name, clf in [
    ('Single Tree (max_depth=3)', DecisionTreeClassifier(max_depth=3, random_state=42)),
    ('Single Tree (unpruned)', DecisionTreeClassifier(random_state=42)),
    ('Random Forest (100 trees)', RandomForestClassifier(n_estimators=100, random_state=42)),
]:
    cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='balanced_accuracy')
    clf.fit(X_train, y_train)
    test_acc = clf.score(X_test, y_test)
    results[name] = {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
                       'test_acc': test_acc, 'clf': clf}
    print(f'{name}: CV={cv_scores.mean():.3f}±{cv_scores.std():.3f}, test={test_acc:.3f}')

# Feature importance
rf = results['Random Forest (100 trees)']['clf']
print('\nRF feature importances:')
for fname, imp in sorted(zip(feature_names, rf.feature_importances_), key=lambda x:-x[1]):
    print(f'  {fname}: {imp:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Decision boundary (tumor size vs. gene A)
ax = axes[0]
xx, yy = np.meshgrid(np.linspace(0, 10, 200), np.linspace(0, 12, 200))
X_grid = np.c_[np.full(xx.ravel().shape, 60), xx.ravel(), np.full(xx.ravel().shape, 3),
                  yy.ravel(), np.full(xx.ravel().shape, 5)]
Z = rf.predict_proba(X_grid)[:, 1].reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.7, vmin=0, vmax=1)
ax.scatter(X_train[y_train==0, 1], X_train[y_train==0, 3], c='steelblue', s=10, alpha=0.5, label='Good')
ax.scatter(X_train[y_train==1, 1], X_train[y_train==1, 3], c='tomato', s=10, alpha=0.5, label='Bad')
ax.set_xlabel('Tumor size')
ax.set_ylabel('Gene A expression')
ax.set_title('A. RF decision boundary\n(blue=good, red=bad prognosis)')
ax.legend(fontsize=8)

# Panel B: Feature importance
ax = axes[1]
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]
ax.barh(range(len(feature_names)), importances[idx][::-1], color='steelblue', edgecolor='black')
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels([feature_names[i] for i in idx[::-1]])
ax.set_xlabel('Mean decrease in impurity')
ax.set_title('B. RF feature importance\n(MDI)')

# Panel C: OOB error vs number of trees
ax = axes[2]
oob_errors = []
for n_trees in range(1, 151, 5):
    rf_tmp = RandomForestClassifier(n_estimators=n_trees, oob_score=True, random_state=42)
    rf_tmp.fit(X_train, y_train)
    oob_errors.append(1 - rf_tmp.oob_score_)
ax.plot(range(1, 151, 5), oob_errors, 'steelblue', lw=2)
ax.set_xlabel('Number of trees')
ax.set_ylabel('OOB error rate')
ax.set_title('C. OOB error vs. n_estimators\n(plateaus after ~50 trees)')
ax.axhline(min(oob_errors), color='red', ls='--', lw=0.8, label=f'Min OOB={min(oob_errors):.3f}')
ax.legend(fontsize=8)

plt.suptitle('Module 13 NB04: Decision Trees and Random Forests', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('random_forests.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement a minimal `DecisionTree` class from scratch: `fit()` (recursive best-split
   + node construction), `predict()` (traverse tree). Verify it matches sklearn's
   `DecisionTreeClassifier` on the clinical dataset.
2. Show the overfitting of an unpruned single tree: plot train accuracy vs. test accuracy
   as `max_depth` increases from 1 to 20.
3. Compare MDI feature importance to permutation importance (`sklearn.inspection.
   permutation_importance`). Do they agree? Which features change rank?
4. Why does the random forest OOB error plateau at ~50 trees? What theoretical
   argument explains this convergence?

---
## Step 10 — Quiz

1. What is the Gini impurity of a node with 50% class 0 and 50% class 1?
2. How does a random forest reduce variance compared to a single decision tree?
3. What is the out-of-bag (OOB) error and why is it useful?
4. What is the typical feature subset size for random forests in classification?
5. What is a limitation of MDI (mean decrease impurity) as a feature importance measure?

---
## Step 12 — Reflection

> *[In one paragraph: explain why bagging reduces variance but not bias. What does
> this imply about when random forests will and won't work well?]*

---
*Next: `05_support_vector_machines.ipynb`*